# 05 — Network Analysis & Career Path Graph

**Owner**: Person B

Build a career path network graph using O*NET Related Occupations
and cosine similarity from the skills matrix. Compute centrality
metrics and run community detection.

### Input files
- `../data/processed/occupation_features.csv` (feature matrix from notebook 01)
- `../data/processed/related_occupations.csv` (O*NET edges from notebook 01)
- `../data/processed/job_zones.csv` (complexity labels)
- Cluster labels from notebook 04 (after Person A completes it)

### Output files
- `../data/processed/career_graph.graphml` (NetworkX graph export)
- `../outputs/career_map.html` (interactive pyvis visualization)
- `../outputs/centrality_table.csv`

In [1]:
import pandas as pd
cluster_df = pd.read_csv('../data/processed/cluster_labels.csv')
print(cluster_df.columns.tolist())
cluster_df.head()

['O*NET-SOC Code', 'Title', 'cluster', 'Job Zone', 'cluster_name']


,O*NET-SOC Code,Title,cluster,Job Zone,cluster_name
0,11-3013.00,Facilities Managers,2,3,Skilled Trades
1,11-9021.00,Construction Managers,0,4,Management/Engineering
2,11-9041.00,Architectural and Engineering Managers,0,5,Management/Engineering
3,17-2051.00,Civil Engineers,0,4,Management/Engineering
4,17-2051.01,Transportation Engineers,0,4,Management/Engineering


In [4]:

from pyvis.network import Network
import pandas as pd

# 3个cluster对应3种颜色
cluster_colors = {
    'Management/Engineering': '#FF5722',
    'Skilled Trades': '#2196F3',
    'Entry Level/Operators': '#4CAF50'
}

cluster_lookup = dict(zip(cluster_df['O*NET-SOC Code'], cluster_df['cluster_name']))

# 重新生成网络图
net2 = Network(height='700px', width='100%', bgcolor='#222222', font_color='white')
net2.barnes_hut()

for code in G.nodes():
    zone = zone_lookup.get(code, 3)
    cluster_name = cluster_lookup.get(code, 'Skilled Trades')
    color = cluster_colors.get(cluster_name, '#888888')
    net2.add_node(
        code,
        label=title_lookup.get(code, code),
        color=color,
        size=zone_sizes.get(zone, 30) * 2,
        title=f"{title_lookup.get(code, code)}<br>Job Zone: {zone}<br>Cluster: {cluster_name}"
    )

for src, dst, data in G.edges(data=True):
    if src == dst:
        continue
    color = '#888888' if data.get('source') == 'onet' else '#444444'
    net2.add_edge(src, dst, color=color, width=1.5 if data.get('source') == 'onet' else 0.5)

net2.save_graph('../outputs/career_map_clusters.html')
print("Saved career_map_clusters.html")

Saved career_map_clusters.html


In [3]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from pyvis.network import Network

# 加载数据
features_df = pd.read_csv('../data/processed/occupation_features.csv')
related_df = pd.read_csv('../data/processed/related_occupations.csv')
job_zones_df = pd.read_csv('../data/processed/job_zones.csv')
cluster_df = pd.read_csv('../data/processed/cluster_labels.csv')

our_codes = features_df['O*NET-SOC Code'].tolist()
title_lookup = dict(zip(features_df['O*NET-SOC Code'], features_df['Title']))
zone_lookup = dict(zip(job_zones_df['O*NET-SOC Code'], job_zones_df['Job Zone']))
cluster_lookup = dict(zip(cluster_df['O*NET-SOC Code'], cluster_df['cluster_name']))

# 计算余弦相似度
feature_cols = [c for c in features_df.columns if c not in ['O*NET-SOC Code', 'Title']]
X = features_df[feature_cols].values
sim_matrix = cosine_similarity(X)

# 构建网络图
G = nx.Graph()
zone_sizes = {1: 20, 2: 25, 3: 30, 4: 35, 5: 40}

for code in our_codes:
    G.add_node(code, title=title_lookup.get(code, code), job_zone=zone_lookup.get(code, 0))

for _, row in related_df.iterrows():
    src = row['O*NET-SOC Code']
    dst = row['Related O*NET-SOC Code']
    if src in our_codes and dst in our_codes:
        G.add_edge(src, dst, source='onet', weight=1.0)

threshold = 0.95
for i, code1 in enumerate(our_codes):
    for j, code2 in enumerate(our_codes):
        if i >= j:
            continue
        if sim_matrix[i][j] >= threshold and not G.has_edge(code1, code2):
            G.add_edge(code1, code2, source='similarity', weight=round(sim_matrix[i][j], 3))

# 生成网络图（按cluster上色）
cluster_colors = {
    'Management/Engineering': '#FF5722',
    'Skilled Trades': '#2196F3',
    'Entry Level/Operators': '#4CAF50'
}

net2 = Network(height='700px', width='100%', bgcolor='#222222', font_color='white')
net2.barnes_hut()

for code in G.nodes():
    zone = zone_lookup.get(code, 3)
    cluster_name = cluster_lookup.get(code, 'Skilled Trades')
    color = cluster_colors.get(cluster_name, '#888888')
    net2.add_node(
        code,
        label=title_lookup.get(code, code),
        color=color,
        size=zone_sizes.get(zone, 30) * 2,
        title=f"{title_lookup.get(code, code)}<br>Job Zone: {zone}<br>Cluster: {cluster_name}"
    )

for src, dst, data in G.edges(data=True):
    if src == dst:
        continue
    color = '#888888' if data.get('source') == 'onet' else '#444444'
    net2.add_edge(src, dst, color=color, width=1.5 if data.get('source') == 'onet' else 0.5)

net2.save_graph('../outputs/career_map_clusters.html')
print("Saved!")

Saved!
